# Fashion-MNIST: IID and non-IID client splits

This notebook downloads the public Fashion-MNIST files, rebuilds the split used in the project with seed 42, and shows the class distribution each client receives. No project files are needed.

In [ ]:
import gzip
import struct
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

CLASSES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
           "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]
CLIENTS = ["c1", "c2", "c3", "c4"]
BASE = "https://raw.githubusercontent.com/zalandoresearch/fashion-mnist/master/data/fashion/"
FILES = {
    "train_images": "train-images-idx3-ubyte.gz",
    "train_labels": "train-labels-idx1-ubyte.gz",
    "test_images": "t10k-images-idx3-ubyte.gz",
    "test_labels": "t10k-labels-idx1-ubyte.gz",
}
RAW = Path("data/raw")
SEED = 42
RAW.mkdir(parents=True, exist_ok=True)

In [ ]:
for name in FILES.values():
    target = RAW / name
    if not target.exists():
        urllib.request.urlretrieve(BASE + name, target)
print(sorted(path.name for path in RAW.glob("*.gz")))

In [ ]:
def load_images(path):
    with gzip.open(path, "rb") as handle:
        data = handle.read()
    _, count, rows, cols = struct.unpack(">IIII", data[:16])
    array = np.frombuffer(data, dtype=np.uint8, count=count * rows * cols, offset=16)
    return array.reshape(count, rows, cols)


def load_labels(path):
    with gzip.open(path, "rb") as handle:
        data = handle.read()
    _, count = struct.unpack(">II", data[:8])
    return np.frombuffer(data, dtype=np.uint8, count=count, offset=8)

In [ ]:
X_train = load_images(RAW / FILES["train_images"])
y_train = load_labels(RAW / FILES["train_labels"])
X_test = load_images(RAW / FILES["test_images"])
y_test = load_labels(RAW / FILES["test_labels"])

print(f"train {X_train.shape}, test {X_test.shape}, dtype {X_train.dtype}")
print(f"pixel mean {X_train.mean():.2f}, std {X_train.std():.2f} on the 0 to 255 scale")
print("train per class", np.bincount(y_train, minlength=10).tolist())
print("test per class", np.bincount(y_test, minlength=10).tolist())

## 1. Dataset

60,000 training images and 10,000 test images, each 28 by 28 pixels, with ten balanced classes.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(11, 5))
for label, ax in enumerate(axes.flat):
    ax.imshow(X_train[np.argmax(y_train == label)], cmap="gray")
    ax.set_title(CLASSES[label], fontsize=8)
    ax.axis("off")
fig.tight_layout()
plt.show()

In [ ]:
counts_train = np.bincount(y_train, minlength=10)
counts_test = np.bincount(y_test, minlength=10)
positions = np.arange(10)

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.bar(positions - 0.2, counts_train, 0.4, label="train")
ax.bar(positions + 0.2, counts_test, 0.4, label="test")
ax.set_xticks(positions, CLASSES, rotation=35, ha="right", fontsize=8)
ax.set_ylabel("images")
ax.set_title("Fashion-MNIST class balance")
ax.legend()
fig.tight_layout()
plt.show()

## 2. IID split

One random permutation of the 60,000 training rows, cut into four equal chunks. Each client should look like the full dataset.

In [ ]:
rng = np.random.default_rng(SEED)
perm = rng.permutation(len(y_train))
iid_chunks = np.array_split(perm, len(CLIENTS))

iid_counts = pd.DataFrame(
    {client: np.bincount(y_train[idx], minlength=10) for client, idx in zip(CLIENTS, iid_chunks)},
    index=CLASSES,
)
print(iid_counts)
print("images per client", iid_counts.sum().tolist())
print("largest deviation from 1500 per class", int((iid_counts - 1500).abs().max().max()))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
bottom = np.zeros(len(CLIENTS))
for label in CLASSES:
    values = iid_counts.loc[label, CLIENTS].to_numpy()
    ax.bar(CLIENTS, values, bottom=bottom, width=0.6, label=label)
    bottom += values
ax.set_ylabel("images")
ax.set_title("IID split")
ax.legend(ncol=5, fontsize=7, loc="upper center", bbox_to_anchor=(0.5, -0.12))
fig.tight_layout()
plt.show()

## 3. non-IID split

Rows sorted by label, then cut into four equal chunks. Each client ends up with two or three classes.

In [ ]:
order = np.argsort(y_train, kind="stable")
noniid_chunks = np.array_split(order, len(CLIENTS))

noniid_counts = pd.DataFrame(
    {client: np.bincount(y_train[idx], minlength=10) for client, idx in zip(CLIENTS, noniid_chunks)},
    index=CLASSES,
)
print(noniid_counts)
print("images per client", noniid_counts.sum().tolist())
for client in CLIENTS:
    present = [CLASSES[i] for i in np.flatnonzero(noniid_counts[client])]
    print(client, present)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
bottom = np.zeros(len(CLIENTS))
for label in CLASSES:
    values = noniid_counts.loc[label, CLIENTS].to_numpy()
    ax.bar(CLIENTS, values, bottom=bottom, width=0.6, label=label)
    bottom += values
ax.set_ylabel("images")
ax.set_title("non-IID split")
ax.legend(ncol=5, fontsize=7, loc="upper center", bbox_to_anchor=(0.5, -0.12))
fig.tight_layout()
plt.show()

## 4. Comparison

Total variation distance from each client to the global class distribution. 0 means identical, 1 means no shared classes.

In [ ]:
global_share = np.bincount(y_train, minlength=10) / len(y_train)


def distance(counts):
    share = counts / counts.sum(axis=0, keepdims=True)
    return 0.5 * np.abs(share - global_share[:, None]).sum(axis=0)


table = pd.DataFrame(
    {"IID": distance(iid_counts.to_numpy()), "non-IID": distance(noniid_counts.to_numpy())},
    index=CLIENTS,
)
print(table.round(3))

assert iid_counts.sum().eq(15000).all()
assert noniid_counts.sum().eq(15000).all()
assert int((iid_counts - 1500).abs().max().max()) < 200
assert int((noniid_counts > 0).sum().max()) <= 3
print("checks passed")